# Revenue Bill Normalization & Consolidation Pipeline

In [43]:
import pandas as pd
import glob
import datetime
import xlsxwriter
import os
pd.set_option('future.no_silent_downcasting', True)

##### To get Current Time

In [44]:
start_time = datetime.datetime.now().strftime("%H:%M:%S")
start_time

'18:33:32'

##### To get Current Date

In [45]:
today=datetime.datetime.now().strftime('%d-%b-%y')
print(today)

09-Jun-26


##### Date-Time

In [46]:
date_time = datetime.datetime.now().strftime("%Y-%m-%d %H-%M-%S")
print(date_time)

2026-06-09 18-33-32


Specifying common path

In [47]:
path = r"D:\Learning\Revenue_Report"

Extracting all files from given path

In [48]:
all_file = glob.glob(path + r"\*.xlsx")

In [49]:
all_file

['D:\\Learning\\Revenue_Report\\Revenue_Report_Dummy_MM_MP.xlsx']

#### Correcting the dateformat in "BillDate column if it contains any AM or PM string"

In [50]:
def convert_dateformat(row):
    if type(row) == str:        
        return pd.to_datetime(row.replace("PM","").replace("AM",""), dayfirst=True)#pd.to_datetime(re.sub(r'[^0-9:]',"",row))#datetime.datetime.strptime(re.sub(r'[^0-9:\s]',"",row),"%d-%m-%Y %H:%M:%S")
    else:
        return row

To Consolidate the files into one Dataframe

In [51]:
def consolidate_reports(all_file, header_row=0, skip_sheets=[]):
    all_df = []
    
    for file in all_file:
        file_n = pd.ExcelFile(file)
        sheet_names = file_n.sheet_names
        sheet_count = len(sheet_names)
        
        data_sheets = [i for i in range(sheet_count) if i not in skip_sheets]
        
        if not data_sheets:
            continue
        
        if len(data_sheets) > 1:
            lst_of_df = []
            headers = None
            
            for i, sheet in enumerate(data_sheets):
                df = pd.read_excel(file, sheet_name=sheet, header=header_row if i == 0 else None)
                if i == 0:
                    headers = df.columns
                else:
                    df.columns = headers[:len(df.columns)]
                lst_of_df.append(df)
            
            df = pd.concat(lst_of_df, ignore_index=True)
        else:
            df = pd.read_excel(file, sheet_name=data_sheets[0], header=header_row)
        all_df.append(df)
    
    if not all_df:
        return pd.DataFrame()
    return pd.concat(all_df, ignore_index=True)
raw_consolidated = consolidate_reports(all_file, header_row=0, skip_sheets=[])

To apply removing " AM" and " PM" vals in datetime value function

In [52]:
raw_consolidated["BILLDATE"] = raw_consolidated["BILLDATE"].map(convert_dateformat)

In [53]:
raw_consolidated.columns

Index(['Unit', 'BILLNO', 'BILLDATE', 'LOCATIONID', 'PATIENTSERVICE',
       'Payer Name1', 'BILLINGTYPENAME', 'PATIENT NAME', 'DOCTORNAME',
       'Bill Amount1', 'Payer Name2', 'Bill Amount2'],
      dtype='object')

In [54]:
raw_consolidated.head(1)

,Unit,BILLNO,BILLDATE,LOCATIONID,PATIENTSERVICE,Payer Name1,BILLINGTYPENAME,PATIENT NAME,DOCTORNAME,Bill Amount1,Payer Name2,Bill Amount2
0,Bhubaneswar,AH-BBS-IP-1000,2026-03-12,11101,IP,MEDI ASSIST-GO DIGIT GENERAL INSURANCE LTD,CREDIT,Jimmy McGill,Dr. Reddy,51515.19,STAR HEALTH INSURANCE CO.LTD,12083.81


Copying the raw_consolidated dataframe for seggregation workings

In [55]:
OS2 = raw_consolidated.copy()

In [56]:
OS2.head(1)

,Unit,BILLNO,BILLDATE,LOCATIONID,PATIENTSERVICE,Payer Name1,BILLINGTYPENAME,PATIENT NAME,DOCTORNAME,Bill Amount1,Payer Name2,Bill Amount2
0,Bhubaneswar,AH-BBS-IP-1000,2026-03-12,11101,IP,MEDI ASSIST-GO DIGIT GENERAL INSURANCE LTD,CREDIT,Jimmy McGill,Dr. Reddy,51515.19,STAR HEALTH INSURANCE CO.LTD,12083.81


Replacing null values with 0

In [57]:
OS2["Bill Amount2"] = OS2["Bill Amount2"].fillna(0)

Filtering rows where OUTSTANDING2 is not 0

In [58]:
OS2 = OS2[OS2["Bill Amount2"]>0]

In [59]:
OS2.head(1)

,Unit,BILLNO,BILLDATE,LOCATIONID,PATIENTSERVICE,Payer Name1,BILLINGTYPENAME,PATIENT NAME,DOCTORNAME,Bill Amount1,Payer Name2,Bill Amount2
0,Bhubaneswar,AH-BBS-IP-1000,2026-03-12,11101,IP,MEDI ASSIST-GO DIGIT GENERAL INSURANCE LTD,CREDIT,Jimmy McGill,Dr. Reddy,51515.19,STAR HEALTH INSURANCE CO.LTD,12083.81


Making selected columns 0. So, the OUTSTANDING2 value can be replaced in future

In [60]:
OS2_cols_to_replaced = ["Bill Amount1"]

In [61]:
OS2[OS2_cols_to_replaced]=0

Viewing the OS2 Dataframe after the replacements

In [62]:
OS2[["BILLNO","Bill Amount1","Bill Amount2"]].head(5)

,BILLNO,Bill Amount1,Bill Amount2
0,AH-BBS-IP-1000,0,12083.81
20,AH-BBS-IP-1004,0,42147.00
22,AH-MUM-IP-1000,0,40058.50
25,AH-BLR-IP-1000,0,26046.00
32,AH-CHN-OP-2000,0,28833.50


For the OS2 Dataframe , adding "-MP" Suffix to bills

In [63]:
OS2["BILLNO"] = OS2["BILLNO"].astype(str) + "-MP"

In [64]:
OS2[["BILLNO","Bill Amount1","Bill Amount2"]].head(5)

,BILLNO,Bill Amount1,Bill Amount2
0,AH-BBS-IP-1000-MP,0,12083.81
20,AH-BBS-IP-1004-MP,0,42147.00
22,AH-MUM-IP-1000-MP,0,40058.50
25,AH-BLR-IP-1000-MP,0,26046.00
32,AH-CHN-OP-2000-MP,0,28833.50


In [65]:
OS2.head(1)

,Unit,BILLNO,BILLDATE,LOCATIONID,PATIENTSERVICE,Payer Name1,BILLINGTYPENAME,PATIENT NAME,DOCTORNAME,Bill Amount1,Payer Name2,Bill Amount2
0,Bhubaneswar,AH-BBS-IP-1000-MP,2026-03-12,11101,IP,MEDI ASSIST-GO DIGIT GENERAL INSURANCE LTD,CREDIT,Jimmy McGill,Dr. Reddy,0,STAR HEALTH INSURANCE CO.LTD,12083.81


Renaming columns in order to make the OS2 dataframe look like original

We can't just rename the columns as OUTSTANDING2 , PAYER2...etc. Because , we are renaming some columns in the same name. and it will violate them.

In [66]:
OS2 = OS2.rename(columns={"Bill Amount1" : "Bill Amount2_new",
                "Bill Amount2" : "Bill Amount1_new", 
                 "Payer Name1" : "Payer Name2_new", 
                 "Payer Name2" : "Payer Name1_new"})

In [67]:
OS2[["Bill Amount2_new","Bill Amount1_new", "Payer Name2_new","Payer Name1_new"]].head()

,Bill Amount2_new,Bill Amount1_new,Payer Name2_new,Payer Name1_new
0,0,12083.81,MEDI ASSIST-GO DIGIT GENERAL INSURANCE LTD,STAR HEALTH INSURANCE CO.LTD
20,0,42147.00,HDFC ERGO GENERAL INSURANCE,BAJAJ FINSERVE
22,0,40058.50,HDFC ERGO GENERAL INSURANCE,INDIAN OIL CORPORATION LIMITED(IOCL)
25,0,26046.00,UNITED INDIA INSURANCE CO LTD,MEDIASSIST TPA (TNNHIS)
32,0,28833.50,ORIENTAL INSURANCE CO LTD,MEDI ASSIST-RELIANCE GENERAL INSURANCE CO LTD


Now, we're safely renaming the columns as per the source

In [68]:
OS2 = OS2.rename(columns={"Bill Amount2_new" : "Bill Amount2",
                "Bill Amount1_new" : "Bill Amount1", 
                 "Payer Name2_new" : "Payer Name2",
                 "Payer Name1_new" : "Payer Name1"})

In [69]:
OS2[["Bill Amount2","Bill Amount1", "Payer Name2"]].head()

,Bill Amount2,Bill Amount1,Payer Name2
0,0,12083.81,MEDI ASSIST-GO DIGIT GENERAL INSURANCE LTD
20,0,42147.00,HDFC ERGO GENERAL INSURANCE
22,0,40058.50,HDFC ERGO GENERAL INSURANCE
25,0,26046.00,UNITED INDIA INSURANCE CO LTD
32,0,28833.50,ORIENTAL INSURANCE CO LTD


In [70]:
OS2.columns

Index(['Unit', 'BILLNO', 'BILLDATE', 'LOCATIONID', 'PATIENTSERVICE',
       'Payer Name2', 'BILLINGTYPENAME', 'PATIENT NAME', 'DOCTORNAME',
       'Bill Amount2', 'Payer Name1', 'Bill Amount1'],
      dtype='object')

To sort the dataframe in Source file order

In [71]:
col_order = ['Unit', 'BILLNO', 'BILLDATE', 'LOCATIONID', 'PATIENTSERVICE',
       'Payer Name1', 'BILLINGTYPENAME', 'PATIENT NAME', 'DOCTORNAME',
       'Bill Amount1', 'Payer Name2', 'Bill Amount2']

In [72]:
OS2.head(1)

,Unit,BILLNO,BILLDATE,LOCATIONID,PATIENTSERVICE,Payer Name2,BILLINGTYPENAME,PATIENT NAME,DOCTORNAME,Bill Amount2,Payer Name1,Bill Amount1
0,Bhubaneswar,AH-BBS-IP-1000-MP,2026-03-12,11101,IP,MEDI ASSIST-GO DIGIT GENERAL INSURANCE LTD,CREDIT,Jimmy McGill,Dr. Reddy,0,STAR HEALTH INSURANCE CO.LTD,12083.81


Applying the sort order

In [73]:
OS2 = OS2[col_order]

In [74]:
OS2.head(1)

,Unit,BILLNO,BILLDATE,LOCATIONID,PATIENTSERVICE,Payer Name1,BILLINGTYPENAME,PATIENT NAME,DOCTORNAME,Bill Amount1,Payer Name2,Bill Amount2
0,Bhubaneswar,AH-BBS-IP-1000-MP,2026-03-12,11101,IP,STAR HEALTH INSURANCE CO.LTD,CREDIT,Jimmy McGill,Dr. Reddy,12083.81,MEDI ASSIST-GO DIGIT GENERAL INSURANCE LTD,0


In [75]:
raw_consolidated.head(1)

,Unit,BILLNO,BILLDATE,LOCATIONID,PATIENTSERVICE,Payer Name1,BILLINGTYPENAME,PATIENT NAME,DOCTORNAME,Bill Amount1,Payer Name2,Bill Amount2
0,Bhubaneswar,AH-BBS-IP-1000,2026-03-12,11101,IP,MEDI ASSIST-GO DIGIT GENERAL INSURANCE LTD,CREDIT,Jimmy McGill,Dr. Reddy,51515.19,STAR HEALTH INSURANCE CO.LTD,12083.81


Checking raw_consolidated (original df) and OS2 df columns are identical.

In [76]:
raw_consolidated.columns == OS2.columns

array([ True,  True,  True,  True,  True,  True,  True,  True,  True,
        True,  True,  True])

Now, Copying the raw_consolidated dataframe in order to merge OS2 dataframe

In [77]:
OS1 = raw_consolidated.copy()

replacing Null with 0

In [78]:
OS1["Bill Amount1"] = OS1["Bill Amount1"].fillna(0)

Removing any values in OUTSTANDING2 Column

And, we're not removing the OS2 related columns in OS1 "Bill Number" for the safeplay. if we want to find what the Bill NNumber's second payer? is this bill having payer2? ...etc , it will help 

In [79]:
OS1["Bill Amount2"]=0

Merging two ( OS1 & OS2 ) dataframes into one

In [80]:
final_consolidated = pd.concat([OS1,OS2],ignore_index=True)

#### Dropping Unnecessary Columns

In [81]:
cols_to_drop = ["Payer Name2","Bill Amount2"]

In [82]:
final_consolidated.drop(columns=cols_to_drop, inplace=True,errors="ignore")

To write the final_consolidated dataframe as an Excel file.

In [83]:
with pd.ExcelWriter(r"D:\Learning\Consolidated MM Revenue with MP Cases"+date_time+".xlsx",engine="xlsxwriter",datetime_format="DD-MMM-YY") as writer:
    final_consolidated.to_excel(writer,sheet_name="Sheet1",index=False)

To write the raw_consolidated (revenue data without any changes) dataframe as an Excel file. (optional)

In [84]:
#with pd.ExcelWriter(r"D:\0) ADD\Revenue_MP\Consolidated MM Revenue raw.xlsx",engine="xlsxwriter",datetime_format="DD-MMM-YY") as writer:
#    raw_consolidated.to_excel(writer,sheet_name="Sheet1",index=False)